In [2]:
import requests

PORT = 8000

END_POINT = 'chat'

url = f'http://localhost:{PORT}/{END_POINT}'

data = "Zuza meduza"

def sent_request(data):

    response = requests.post(url, json = data)

    if response == 200:
        print(response.json())
    else:
        print(response)

sent_request(data)

# response = requests.get(url)

# print(response, response.text)



<Response [422]>


In [ ]:
import requests

PORT = 8000

END_POINT = 'chat/'

url = f'http://localhost:{PORT}/{END_POINT}'

data = {'prompt': 'Bombardiro Crocodilo'}

response = requests.post(url, data=data, stream=True)  # stream=True jeśli chcesz strumieniować odpowiedź
for chunk in response.iter_content(chunk_size=1024):
    if chunk:
        print(chunk.decode(), end='')

In [13]:
import requests


PORT = 8000

END_POINT = '/add_message'
# END_POINT = '/chat'

url = f'http://localhost:{PORT}/{END_POINT}'

data = {'prompt': 'Ala ma kruka!'}

response = requests.post(url, data=data)  # stream=True jeśli chcesz strumieniować odpowiedź
# for chunk in response.iter_content(chunk_size=1024):
#     if chunk:
#         print(chunk.decode(), end='')

In [21]:
# CHECK DB Content

import sqlite3

# Path to your .sqlite file
db_path = "chat_app_messages.sqlite"

# Connect to the SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Get all table names (optional, for exploration)
cursor.execute("SELECT * FROM messages;")

rows = cursor.fetchall()

# Clean up
cursor.close()
conn.close()


In [20]:
rows

[(None,
  b'[{"parts":[{"content":"test","timestamp":"2025-05-21T20:02:58.644846Z","part_kind":"user-prompt"}],"instructions":null,"kind":"request"},{"parts":[{"content":"Hello! How can I assist you today?","part_kind":"text"}],"model_name":"gpt-3.5-turbo","timestamp":"2025-05-21T20:02:59Z","kind":"response"}]'),
 (None,
  b'[{"parts":[{"content":"test","timestamp":"2025-05-21T21:02:13.052559Z","part_kind":"user-prompt"}],"instructions":null,"kind":"request"},{"parts":[{"content":"All good on my end! Let me know if you have any questions or need help with anything.","part_kind":"text"}],"model_name":"gpt-3.5-turbo","timestamp":"2025-05-21T21:02:15Z","kind":"response"}]'),
 (None,
  b'[{"parts":[{"content":"You are a DevOps expert. Your task is to analyze log messages received from Kafka and provide concise, \\n                    actionable explanations or solutions for any detected issues. Focus on identifying errors, warnings, and \\n                    abnormal patterns.","timestamp

In [ ]:
from fastapi import FastAPI, Request
from pydantic import BaseModel
from pydantic_ai import Agent

app = FastAPI()

# Define the request schema
class QueryRequest(BaseModel):
    message: str

# Initialize the LLM agent
agent = Agent(model="gpt-4", system="You are a helpful assistant.")

@app.post("/chat")
async def chat(request: QueryRequest):
    response = await agent.run(request.message)
    return {"response": response}


In [14]:
# DISCORD CHANEL

webhook_id = 'FzUvToXRdk0WtCqrZhcwIgmEu-R-hoSHQc8eyFsT6OBGX8_n-zPzjNgDouDKocGlNX-w'
webhook_url = f"https://discord.com/api/webhooks/1377369991803834391/{webhook_id}"


import requests

def send_to_discord(webhook_url, message):
    requests.post(webhook_url, json={"content": message})


send_to_discord(webhook_url, "XYZ")

In [1]:

import redis

# Connect to Redis
r = redis.Redis(host='localhost', port=6379, db=0)

# Zapisz coś z TTL (np. 15 minut = 900 sekund)
r.set("log:example", "vector_string_or_serialized_data", ex=900)

# Odczytaj
value = r.get("log:example")

print("Z Redis:", value.decode() if value else "Brak klucza")


Z Redis: vector_string_or_serialized_data


In [39]:
# REDIS TESTS:

from pydantic import BaseModel
from redis import Redis
import time
import uuid
from typing import Tuple

# class LogEntry(BaseModel):
#     message: str

# Connect to Redis:
redis_db = Redis(host='localhost', port=6379, db=0)
# db=0 means default Redis logical database (database number 0).

# How long logs will stay in Redis db
LOG_TTL = 15 * 60  # 15 minutes TTL

def make_red_log_id() -> Tuple[int, str]:
    """ Function to generate log id in Redis DB"""

    # millisecond timestamp
    ms_time_stamp = int(time.time() * 1000)

    # short UUID suffix to guarantee uniqueness
    uuid_suffix = uuid.uuid4().hex[:6]

    return (ms_time_stamp, uuid_suffix)


def store_log_redis(entry):
    """store log in Redis DB"""

    # Create log
    redis_log_id = make_red_log_id()

    # timestamp in ms:
    ms_time_stamp: int = redis_log_id[0]
    suffix_id: str = redis_log_id[1]

    # Store log message with TTL
    redis_db.set(f"{ms_time_stamp}:{suffix_id}", entry, ex=LOG_TTL)

    # Add to sorted set with timestamp as score
    redis_db.zadd("logs:zset", {suffix_id: ms_time_stamp})

    return {"redis_log_id": redis_log_id, "redis_timestamp": ms_time_stamp}

log1 = '[2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)'

store_log_redis(log1)

{'redis_log_id': (1748551659648, '665fa1'), 'redis_timestamp': 1748551659648}

In [ ]:
def get_context_logs(log_id: str):
    """Return 5 logs before given log"""

    rank = redis_db.zrank("logs:zset", log_id)
    
    if rank is None or rank == 0:
        return []

    start = max(0, rank - 5)
    prev_log_ids = redis_db.zrange("logs:zset", start, rank - 1)

    logs = []
    for lid in prev_log_ids:
        lid_str = lid.decode()
        msg = redis_db.get(f"log:{lid_str}")

        if msg:
            logs.append({"log_id": lid_str, "message": msg.decode()})

    return logs[::-1]  # newest first







[]

In [45]:
# return simple ID:

def get_context_logs(log_id: str):
    """Return given log"""

    given_log = redis_db.get(f"{log_id}")

    return given_log  # newest first

idX = (1748551659648, '665fa1')

res = get_context_logs(f'{idX[0]}:{idX[1]}').decode()

res

'[2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)'

In [40]:
keys = redis_db.keys("*")

print("All keys:")

for key in keys:
    print(key.decode())

All keys:
1748551655690:dc1893
log:(1748550906418, '1e16f2')
1748551654658:7bec10
log:(1748550904109, 'fa7d52')
1748551656041:d581eb
1748551655079:a2fbdc
XXX:(1748551579806, '0b8dd1')
log:(1748550907041, 'bd1c03')
log:(1748550905503, '461756')
1748551659648:665fa1
log:(1748550906775, '501d34')
log:(1748550906280, '4a9a8a')
1748551656795:23e1da
log:(1748550906128, '076f4e')
log:(1748550905688, 'f54a46')
logs:zset
log:(1748550905837, '970f54')
XXX:(1748551580643, '0db713')
log:(1748550907162, '310f91')
log:(1748550906918, '12c6da')
log:(1748550905989, '1fdf13')
1748551658956:b8babe
log:(1748550906668, '2ea129')
log:(1748550905030, '68f8d4')
1748551658204:e2bab8
1748551656383:ade31a
log:(1748550906520, 'da6ba0')
log:(1748550907299, 'ea6fa2')
XXX:(1748551582447, '2ce8f6')


In [ ]:
keys = redis_db.keys("*")

print("All keys and values:")

for key in keys:

    key_str = key.decode()

    key_type = redis_db.type(key).decode()
    

    print(f"\nKey: {key_str} (Type: {key_type})")

    if key_type == "string":

        value = redis_db.get(key)

        print(f"  Value: {value.decode()}")

    elif key_type == "zset":

        entries = redis_db.zrange(key, 0, -1, withscores=True)

        for member, score in entries:
            print(f"  Member: {member.decode()} | Score: {int(score)}")

    else:
        print("[Unhandled or empty value]")


All keys and values:

Key: log:(1748550441537, 'f447c4') (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: log:(1748550906418, '1e16f2') (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: log:(1748550437079, '621b1a') (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: log:(1748550438322, '31bed4') (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: log:(1748550440951, '15cc9b') (Type: string)
  Value: [2025-04-25 22:35:18,082] INFO Registered signal handlers for TERM, INT, HUP (org.apache.kafka.common.utils.LoggingSignalHandler)

Key: log:(1748550440352, '48